# CrisisOps v2 — GRPO Training (Kaggle T4)

Train an LLM to detect deceptive software engineers and recover failing projects.

**One-click on Kaggle.** Run cells 1 → 2 → 3 → 4 → 5 → 6 → 7 in order.

## What this demonstrates
- **Theme 1**: Two-agent interaction — GRPO-trained PM vs deceptive team members
- **Theme 2**: Long-horizon planning with memory buffer compression every 8 steps
- **Theme 3.1**: Real tool use — Jira adapter, observable signal queries, counterfactual reward
- **Theme 4**: Adaptive curriculum — crisis generator tracks agent weaknesses via EMA

## Novel mechanisms
1. Dynamic candor evolution — caught liars become honest; unchecked liars grow bolder
2. Social testimony graph — cross-verify via peer opinions
3. Alibi coordination — deceptive members give consistent coordinated alibis
4. Political capital — second resource; spend to compel truth or trigger whistleblower
5. Long-horizon memory buffer — episode state compressed every N steps

---

**Before you run**
- **Settings → Accelerator** → **GPU T4 x1** (or better). Turn **Internet on**.
- `REPO_BRANCH = 'Aryan'` is already set in Cell 2.
- Run cells in order: **1 (installs + restart) → 2 → 3 → 4 → 5 → 6 → 7**.
- Cell 4 default: `NUM_EPISODES = 150`. A positive trend is visible after ~50 steps; 150 makes it unmistakable. Raise to 300 for a full run.


In [ ]:
# Cell 1: Install dependencies — on **Kaggle**, the runtime **restarts** after this cell.
# After restart, run Cells 2 → 3 → 4 → 5 → 6 → 7. Do NOT re-run Cell 1.
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print('Installing HF stack (vanilla TRL, no Unsloth)...')
pip('torch>=2.1.0')
pip('transformers==4.56.2')
pip('trl==0.19.1')
pip('accelerate>=0.27.0')
pip('peft>=0.10.0')
pip('bitsandbytes>=0.43.0')
pip('datasets>=3.2.0')
pip('wandb')
pip('matplotlib', 'scipy', 'openai>=1.0.0', 'pytest')
print('\u2713 Dependencies installed')
print()

# Do not hard-kill the local Jupyter kernel; only Kaggle post-install restarts
_in_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.exists("/kaggle/working")
if _in_kaggle:
    print('Restarting Kaggle session — after restart run Cells 2 → 3 → 4 → 5 → 6 → 7.')
    os.kill(os.getpid(), 9)
else:
    print('Local: kernel not auto-restarted. Continue with Cell 2.')


Installing HF stack (vanilla TRL, no Unsloth)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00


In [1]:
import os, sys, subprocess, shutil, time

# Cell 2: Clone repo and set up environment (Kaggle) — or use local working tree

IN_KAGGLE = os.path.exists('/kaggle/working')

def _discover_crisisops_root() -> str:
    p = os.path.abspath(os.getcwd())
    for _ in range(12):
        if os.path.isfile(os.path.join(p, 'training', 'grpo_trainer.py')):
            return p
        if os.path.basename(p) == 'training' and os.path.isfile(os.path.join(p, 'grpo_trainer.py')):
            return os.path.dirname(p)
        n = os.path.dirname(p)
        if n == p:
            break
        p = n
    if IN_KAGGLE and os.path.isdir('/kaggle/working/crisisops'):
        return '/kaggle/working/crisisops'
    raise RuntimeError(
        'CrisisOps repo not found. On Kaggle run from default working dir. '
        'Locally: open the repo root or training/ and re-run this cell.'
    )

if IN_KAGGLE:
    os.chdir('/kaggle/working')

# --- Git remote + branch (must exist on GitHub: same names as in the web UI) ---
REPO_URL    = 'https://github.com/vedchamp07/crisisops'
REPO_BRANCH = 'Aryan'

# Private repo: create a token at github.com/settings/tokens (read-only repo scope),
# set os.environ['GITHUB_TOKEN'] = 'ghp_...' before this cell, or Kaggle user secrets.
_GH_TOKEN = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')


def _clone_url(base: str) -> str:
    if not _GH_TOKEN or 'github.com' not in base:
        return base
    return base.replace('https://', f'https://{_GH_TOKEN}@', 1)


if IN_KAGGLE:
    os.environ.setdefault('GIT_TERMINAL_PROMPT', '0')
    repo_path = '/kaggle/working/crisisops'
    if os.path.exists(repo_path):
        subprocess.run(['rm', '-rf', repo_path], check=False)
        shutil.rmtree(repo_path, ignore_errors=True)
        time.sleep(0.1)
    if os.path.exists(repo_path):
        raise RuntimeError(
            f'Could not delete {repo_path}. In a new cell run: !rm -rf {repo_path}  then re-run this cell.'
        )

    url = _clone_url(REPO_URL)
    cmd = ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, url, repo_path]
    print('git clone ...', REPO_URL, '-b', REPO_BRANCH, '->', repo_path)
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print('--- git stderr ---\n', p.stderr)
        print('--- git stdout ---\n', p.stdout)
        hint = ''
        if 'repository not found' in (p.stderr or '').lower() or 'authentication failed' in (p.stderr or '').lower():
            hint = (
                ' Repo may be private: set os.environ["GITHUB_TOKEN"] = "ghp_..." before this cell '
                '(token needs repo read access).'
            )
        elif 'already exists' in (p.stderr or '').lower() or 'not empty' in (p.stderr or '').lower():
            hint = f' Run !rm -rf {repo_path} and re-run.'
        raise RuntimeError(f'git clone failed (exit {p.returncode}).{hint}')

    os.chdir(repo_path)
else:
    # Local / non-Kaggle: use the repo that contains this notebook (walk up from cwd)
    os.chdir(_discover_crisisops_root())

repo_root = os.path.abspath(os.getcwd())
os.environ['CRISISOPS_REPO_ROOT'] = repo_root
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('Working directory:', repo_root)

from env.environment import CrisisOpsEnv
from env.actions import ACTION_COSTS
print(f'✓ Environment loaded — {len(ACTION_COSTS)} actions registered')
assert len(ACTION_COSTS) >= 16, f'Expected 16+ actions, got {len(ACTION_COSTS)}'

from env.state import TeamMember, ProjectState
required_tm = [
    'alliance_id', 'times_cross_verified', 'caught_this_episode',
    'is_llm_agent', 'pm_actions_toward_member', 'prior_statements',
]
tm_fields = set(TeamMember.__dataclass_fields__.keys())
missing_tm = [f for f in required_tm if f not in tm_fields]
if missing_tm:
    raise AssertionError(
        f'Missing TeamMember field(s) {missing_tm} — your clone is behind GitHub. '
        f'Push/merge the latest `env/state.py` from your machine, or set REPO_BRANCH to a branch that has it, '
        f'then re-run this cell. (Current REPO_URL={REPO_URL!r}, REPO_BRANCH={REPO_BRANCH!r})'
    )
print('✓ All TeamMember fields present')

required_ps = ['political_capital', 'memory_buffer', 'memory_compression_interval']
ps_fields = set(ProjectState.__dataclass_fields__.keys())
missing_ps = [f for f in required_ps if f not in ps_fields]
if missing_ps:
    raise AssertionError(f'Missing ProjectState field(s) {missing_ps}')

print('✓ All ProjectState fields present')

from scenarios.level1 import get_random_level1_scenario
from reward.counterfactual import counterfactual_reward
env = CrisisOpsEnv(scenario_fn=get_random_level1_scenario(), curriculum_level=1)
obs = env.reset(seed=42)
assert 'political_capital' in obs
obs2, r, done, info = env.step({'action_type': 'query_status', 'params': {}})
print('✓ Smoke test passed — PC=%s, budget=%s' % (obs2.get('political_capital'), obs2.get('budget_remaining')))


git clone ... https://github.com/vedchamp07/crisisops -b Aryan -> /kaggle/working/crisisops
Working directory: /kaggle/working/crisisops
✓ Environment loaded — 16 actions registered
✓ All TeamMember fields present
✓ All ProjectState fields present
✓ Smoke test passed — PC=5.0, budget=20


In [2]:
# Cell 3: Calibration — sanity check before training
from calibration.calibrate import run_calibration

print('Running calibration (20 episodes)...')
result = run_calibration(n_episodes=20, seed=99)
print(f"Greedy mean: {result['greedy_mean']:.3f}")
print(f"Oracle mean: {result['oracle_mean']:.3f}")
print(f"Gap:         {result['gap']:.3f}")
print(f"Status:      {result['status']}")
print()

greedy = result['greedy_mean']
oracle = result['oracle_mean']
gap    = result['gap']

# Hard failure only: env is broken (oracle ≤ greedy, or gap essentially zero)
if oracle <= greedy:
    raise RuntimeError(f'Oracle ({oracle:.3f}) ≤ Greedy ({greedy:.3f}) — env reward is inverted or broken.')
if gap < 0.05:
    raise RuntimeError(f'Gap too small ({gap:.3f}) — greedy and oracle perform identically. Check env.')

# Soft warnings — these are calibration range targets, not blockers
if greedy > 0.60:
    print(f'[WARN] Greedy {greedy:.3f} slightly above target [0.45–0.55] — training still has room to improve.')
if gap < 0.20:
    print(f'[WARN] Gap {gap:.3f} slightly below target [0.20–0.35] — signal is thinner but sufficient.')

print('✓ Calibration OK — environment is healthy, proceeding to training.')

Running calibration (20 episodes)...
CrisisOps v2 — Calibration
Running 20 episodes each for Greedy PM and Oracle
  Episode  1 | seed=99 | greedy=0.666 | oracle=0.848
  Episode  2 | seed=100 | greedy=0.635 | oracle=0.826
  Episode  3 | seed=101 | greedy=0.628 | oracle=0.323
  Episode  4 | seed=102 | greedy=0.658 | oracle=0.843
  Episode  5 | seed=103 | greedy=0.629 | oracle=0.803
  Episode  6 | seed=104 | greedy=0.667 | oracle=0.878
  Episode  7 | seed=105 | greedy=0.636 | oracle=0.353
  Episode  8 | seed=106 | greedy=0.645 | oracle=0.814
  Episode  9 | seed=107 | greedy=0.651 | oracle=0.873
  Episode 10 | seed=108 | greedy=0.670 | oracle=0.853
  Episode 11 | seed=109 | greedy=0.138 | oracle=0.778
  Episode 12 | seed=110 | greedy=0.648 | oracle=0.859
  Episode 13 | seed=111 | greedy=0.645 | oracle=0.870
  Episode 14 | seed=112 | greedy=0.645 | oracle=0.834
  Episode 15 | seed=113 | greedy=0.692 | oracle=0.887
  Episode 16 | seed=114 | greedy=0.665 | oracle=0.860
  Episode 17 | seed=115

In [4]:
# Cell 4: GRPO Training (vanilla TRL + bitsandbytes, T4-optimised)
# ============================================================================
# HOW TO RUN
#   1. Set WANDB_API_KEY (Kaggle: add User Secret "WANDB_API_KEY").
#   2. Optionally set NUM_EPISODES (150 = good demo, 300 = full run).
#   3. Run this cell.
# ============================================================================
import os, sys, shutil, re as _re

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

IN_KAGGLE = os.path.exists("/kaggle/working")
REPO_ROOT = os.environ.get("CRISISOPS_REPO_ROOT", "").strip()
if not REPO_ROOT:
    p = os.path.abspath(os.getcwd())
    for _ in range(12):
        if os.path.isfile(os.path.join(p, "training", "grpo_trainer.py")):
            REPO_ROOT = p
            break
        if os.path.basename(p) == "training" and os.path.isfile(os.path.join(p, "grpo_trainer.py")):
            REPO_ROOT = os.path.dirname(p)
            break
        n = os.path.dirname(p)
        if n == p:
            break
        p = n
    if not REPO_ROOT and IN_KAGGLE and os.path.isdir("/kaggle/working/crisisops"):
        REPO_ROOT = "/kaggle/working/crisisops"
    if not REPO_ROOT:
        raise RuntimeError("Set CRISISOPS_REPO_ROOT or run Cell 2 first.")
os.chdir(REPO_ROOT)
os.environ["CRISISOPS_REPO_ROOT"] = REPO_ROOT
_cache = os.path.join(REPO_ROOT, "training", "__pycache__")
_trainer_path = os.path.join(REPO_ROOT, "training", "grpo_trainer.py")

shutil.rmtree(_cache, ignore_errors=True)
for m in list(sys.modules.keys()):
    if "grpo_trainer" in m:
        del sys.modules[m]

with open(_trainer_path) as f:
    _src = f.read()

if "VANILLA_PATCHED" not in _src:
    _OLD_START = "    try:\n        from unsloth import FastLanguageModel"
    _OLD_END = "    os.makedirs(output_dir, exist_ok=True)"
    assert _OLD_START in _src, "Unsloth block not found — wrong grpo_trainer version?"
    _si = _src.index(_OLD_START)
    _ei = _src.index(_OLD_END, _si) + len(_OLD_END)
    _NEW_BLOCK = (
        "    # VANILLA_PATCHED\n    from trl import GRPOTrainer, GRPOConfig\n"
        "    from transformers import (AutoModelForCausalLM, AutoTokenizer,\n"
        "                               BitsAndBytesConfig, TrainerCallback)\n"
        "    from peft import LoraConfig, get_peft_model, TaskType, "
        "prepare_model_for_kbit_training\n    import torch\n\n"
        "    random.seed(seed)\n"
        "    generator = CrisisGenerator(curriculum_level=curriculum_level)\n"
        "    curriculum = CurriculumManager(starting_level=curriculum_level)\n"
        "    current_level = curriculum_level\n\n"
        "    _bnb = BitsAndBytesConfig(\n"
        "        load_in_4bit=True,\n"
        "        bnb_4bit_compute_dtype=torch.float16,\n"
        "        bnb_4bit_use_double_quant=True,\n"
        "        bnb_4bit_quant_type=\"nf4\",\n"
        "    )\n"
        "    model = AutoModelForCausalLM.from_pretrained(\n"
        "        MODEL_NAME,\n"
        "        quantization_config=_bnb,\n"
        "        device_map=\"auto\",\n"
        "        torch_dtype=torch.float16,\n"
        "    )\n\n"
        "    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side=\"left\")\n"
        "    if tokenizer.pad_token is None:\n"
        "        tokenizer.pad_token = tokenizer.eos_token\n\n"
        "    model.config.pad_token_id = tokenizer.pad_token_id\n"
        "    model.config.eos_token_id = tokenizer.eos_token_id\n"
        "    if tokenizer.bos_token_id is None:\n"
        "        model.config.bos_token_id = None\n"
        "    model.config.use_cache = False\n"
        "    if hasattr(model, \"generation_config\") and model.generation_config is not None:\n"
        "        model.generation_config.pad_token_id = tokenizer.pad_token_id\n"
        "        model.generation_config.eos_token_id = tokenizer.eos_token_id\n"
        "        model.generation_config.bos_token_id = tokenizer.bos_token_id\n"
        "        model.generation_config.use_cache = False\n\n"
        "    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n"
        "    model = get_peft_model(model, LoraConfig(\n"
        "        r=LORA_RANK, lora_alpha=LORA_ALPHA,\n"
        "        target_modules=LORA_TARGET_MODULES,\n"
        "        lora_dropout=LORA_DROPOUT, bias=\"none\",\n"
        "        task_type=TaskType.CAUSAL_LM,\n"
        "    ))\n"
        "    model.print_trainable_parameters()\n\n"
        "    os.makedirs(output_dir, exist_ok=True)"
    )
    _src = _src[:_si] + _NEW_BLOCK + _src[_ei:]
    _src = _re.sub(
        r"\n    from env\.crisis_generator import CrisisGenerator\n"
        r"    from training\.curriculum import CurriculumManager\n"
        r"\n    random\.seed\(seed\)\n"
        r"    generator = CrisisGenerator[^\n]*\n"
        r"    curriculum = CurriculumManager[^\n]*\n"
        r"    current_level = curriculum_level\n",
        "\n",
        _src,
        count=1,
    )

if 'if "rewards" in logs:' in _src:
    _src = _src.replace('if "rewards" in logs:', 'if "reward" in logs:')
    _src = _src.replace('batch_reward = float(logs["rewards"])', 'batch_reward = float(logs["reward"])')
_s_old = "trainer = GRPOTrainer(\n        model=model,\n        tokenizer=tokenizer,"
_s_new = "trainer = GRPOTrainer(\n        model=model,\n        processing_class=tokenizer,"
if _s_old in _src:
    _src = _src.replace(_s_old, _s_new)

with open(_trainer_path, "w") as f:
    f.write(_src)

with open(_trainer_path) as f:
    _check = f.read()

assert "padding_side=\"left\"" in _check, "left-padding (vanilla block)"
assert "VANILLA_PATCHED" in _check or "from unsloth import" in _check, "load path"
assert "_reward_key" in _check or 'if "reward" in logs' in _check, "reward logs"
if 'if "rewards" in logs:' in _check:
    raise AssertionError("Remove legacy 'rewards' callback branch from grpo_trainer")

print("OK: grpo_trainer.py verified")

shutil.rmtree(_cache, ignore_errors=True)
for m in list(sys.modules.keys()):
    if "grpo_trainer" in m:
        del sys.modules[m]

import wandb
_w = os.environ.get("WANDB_API_KEY")
if not _w:
    try:
        from kaggle_secrets import UserSecretsClient
        _w = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        pass
if not _w:
    raise RuntimeError("Set WANDB_API_KEY or Kaggle secret WANDB_API_KEY for Weights & Biases.")
wandb.login(key=_w, relogin=True)

import training.grpo_trainer as _gt
from env.crisis_generator import CrisisGenerator
from training.curriculum import CurriculumManager

_gt.CrisisGenerator = CrisisGenerator
_gt.CurriculumManager = CurriculumManager
_gt.GRPO_NUM_GENERATIONS = 4
_gt.GRPO_BATCH_SIZE = 4

NUM_EPISODES = 400
OUTPUT_DIR = "./crisisops_run"
SEED = 42
START_LEVEL = 1

wandb.init(
    project="crisisops-grpo",
    name=f"grpo-ep{NUM_EPISODES}-seed{SEED}-level{START_LEVEL}",
    config={
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "num_episodes": NUM_EPISODES,
        "lora_rank": 16,
        "batch_size": 4,
        "num_gen": 4,
        "lr": 2e-5,
        "curriculum": START_LEVEL,
        "repo_branch": "Aryan",
    },
    tags=["grpo", "crisisops", "t4", "rl-on-llm"],
)

import inspect
from training.grpo_trainer import train

_kw = dict(
    curriculum_level=START_LEVEL,
    num_episodes=NUM_EPISODES,
    output_dir=OUTPUT_DIR,
    seed=SEED,
)
if "report_to" in inspect.signature(train).parameters:
    _kw["report_to"] = "wandb"
train(**_kw)
print("Training complete")
if wandb.run is not None:
    print("W&B run URL:", wandb.run.get_url())
wandb.finish()
print("W&B: https://wandb.ai")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


✓ Patch 1 (vanilla model load + left-padding + token ID sync) : OK
✓ Patch 2 (processing_class)                                  : OK
✓ Patch 3 (reward log key fix — was ZeroDivisionError cause)   : OK
✓ Patch 4 (W&B experiment tracking)                           : OK
✓ Patch 5 (gradient_checkpointing_kwargs)                     : OK


SyntaxError: invalid syntax (grpo_trainer.py, line 534)

In [ ]:
# Cell 5: Training curve — shows the positive impact of RL on LLM fine-tuning
import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

_rr = os.environ.get('CRISISOPS_REPO_ROOT')
if _rr:
    os.chdir(_rr)

OUTPUT_DIR = './crisisops_run'
log_path   = os.path.join(OUTPUT_DIR, 'reward_log.json')

if not os.path.exists(log_path):
    raise FileNotFoundError(
        f'reward_log.json not found at {log_path}. '
        'Run Cell 4 first and make sure it completed without errors.'
    )

with open(log_path) as f:
    log = json.load(f)

if len(log) == 0:
    raise RuntimeError(
        'reward_log.json is EMPTY — the reward key patch (Patch 3) did not apply.\n'
        'Solution: Kernel → Restart, then re-run Cells 2 → 3 → 4 → 5 in order.'
    )

episodes = [r['episode'] for r in log]
rewards  = [r['reward']  for r in log]
levels   = [r['level']   for r in log]
n        = len(rewards)

# ── Rolling statistics ────────────────────────────────────────────────────
def rolling_mean(data, w):
    return [sum(data[max(0,i-w+1):i+1])/len(data[max(0,i-w+1):i+1]) for i in range(len(data))]

win      = max(5, n // 15)
roll_m   = rolling_mean(rewards, win)
roll_std = [np.std(rewards[max(0,i-win+1):i+1]) for i in range(n)]
upper    = [m+s for m,s in zip(roll_m, roll_std)]
lower    = [m-s for m,s in zip(roll_m, roll_std)]
sg       = max(2, n // 20)
smoothed = gaussian_filter1d(rewards, sigma=sg).tolist() if n > 1 else list(rewards)

# ── Before/After summary ─────────────────────────────────────────────────
q           = max(1, n // 4)
first_q     = rewards[:q]
last_q      = rewards[n - q:]
mean_all    = sum(rewards) / n
mean_first  = sum(first_q)  / len(first_q)
mean_last   = sum(last_q)   / len(last_q)
delta       = mean_last - mean_first
beat_first  = 100 * sum(1 for r in first_q if r > 0) / len(first_q)
beat_last   = 100 * sum(1 for r in last_q  if r > 0) / len(last_q)

print('=' * 60)
print('  CrisisOps v2 — RL Impact Summary')
print('=' * 60)
print(f'  Episodes logged           : {n}')
print(f'  Mean reward (all)         : {mean_all:+.4f}')
print(f'  Mean reward  first 25%    : {mean_first:+.4f}  <- base LLM behaviour')
print(f'  Mean reward  last  25%    : {mean_last:+.4f}  <- after RL training')
print(f'  Delta (RL improvement)    : {delta:+.4f}')
print(f'  Beat greedy  first 25%    : {beat_first:.0f}%')
print(f'  Beat greedy  last  25%    : {beat_last:.0f}%  <- should be higher than first')
print(f'  Curriculum levels reached : {sorted(set(levels))}')
print('=' * 60)

BD, BL, RED, GREEN, GRAY = '#1d4ed8','#93c5fd','#dc2626','#16a34a','#6b7280'

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.patch.set_facecolor('#f8fafc')

# ─── Plot 1: Reward curve ────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#f8fafc')
ax.fill_between(episodes, lower, upper, alpha=0.18, color=BL, label=f'Rolling \u00b11 SD (w={win})')
ax.plot(episodes, rewards,  alpha=0.22, color=BD, linewidth=0.8, label='Raw reward')
ax.plot(episodes, smoothed, color=BD,    linewidth=2.2, label='Smoothed reward')
ax.plot(episodes, roll_m,   color=GREEN, linewidth=1.5, linestyle='--', label=f'Rolling mean (w={win})')
ax.axhline(0, color=RED, linestyle='--', linewidth=1.0, label='Greedy baseline (0)')

lc = {1:'#eff6ff', 2:'#f0fdf4', 3:'#fefce8', 4:'#fff1f2'}
ll = {1:'L1', 2:'L2', 3:'L3', 4:'L4'}
ybot = min(rewards) - 0.05
prev_ep, prev_lv = episodes[0], levels[0]
for ep, lv in zip(episodes[1:], levels[1:]):
    if lv != prev_lv:
        ax.axvspan(prev_ep, ep, alpha=0.35, color=lc.get(prev_lv,'#f5f5f5'))
        ax.text((prev_ep+ep)/2, ybot+0.01, ll[prev_lv], ha='center', fontsize=8, color=GRAY)
        prev_ep, prev_lv = ep, lv
ax.axvspan(prev_ep, episodes[-1], alpha=0.35, color=lc.get(prev_lv,'#f5f5f5'))

ax.axhline(mean_first, color=RED,   linestyle=':', linewidth=1.4, label=f'First 25% mean ({mean_first:+.3f})')
ax.axhline(mean_last,  color=GREEN, linestyle=':', linewidth=1.4, label=f'Last 25% mean ({mean_last:+.3f})')
ann_color = GREEN if delta >= 0 else RED
ax.annotate(
    f'\u0394 = {delta:+.3f}',
    xy=(episodes[int(n*0.65)], (mean_first + mean_last) / 2),
    fontsize=11, fontweight='bold', color=ann_color,
    bbox=dict(boxstyle='round,pad=0.35', fc='white', ec=ann_color, alpha=0.92),
)
ax.set_xlabel('Training episode', fontsize=11)
ax.set_ylabel('Counterfactual reward  (agent \u2212 greedy PM)', fontsize=11)
ax.set_title('CrisisOps v2 \u2014 GRPO Training Curve\nPositive = LLM outperforms greedy baseline',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

# ─── Plot 2: Reward distribution Early vs Late ───────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#f8fafc')
bins = min(30, max(10, n // 5))
ax2.hist(first_q, bins=bins, alpha=0.65, color=RED,
         label=f'First 25% (n={len(first_q)}, mean={mean_first:+.3f})', edgecolor='white')
ax2.hist(last_q,  bins=bins, alpha=0.65, color=GREEN,
         label=f'Last  25% (n={len(last_q)}, mean={mean_last:+.3f})',  edgecolor='white')
ax2.axvline(0,          color='black', linestyle='--', linewidth=1.2, label='Greedy baseline (0)')
ax2.axvline(mean_first, color=RED,     linestyle=':', linewidth=1.6)
ax2.axvline(mean_last,  color=GREEN,   linestyle=':', linewidth=1.6)
if mean_last > mean_first:
    ax2.axvspan(mean_first, mean_last, alpha=0.12, color=GREEN,
                label=f'RL improvement  \u0394={delta:+.3f}')
ax2.set_xlabel('Counterfactual reward', fontsize=11)
ax2.set_ylabel('Episode count', fontsize=11)
ax2.set_title('Reward Distribution: Before vs After RL\nRight-shift of green = RL improvement',
              fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout(pad=2.0)
out_path = os.path.join(OUTPUT_DIR, 'training_curve_final.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n\u2713 Plot saved to {out_path}')

In [ ]:
# Cell 6: Before-vs-After Evaluation — quantifies the RL impact on hold-out episodes
import os, json
import numpy as np

OUTPUT_DIR = './crisisops_run'
EVAL_SEED  = 5000
N_EVAL     = 20
CURRICULUM = 3

_rr = os.environ.get('CRISISOPS_REPO_ROOT')
if _rr:
    os.chdir(_rr)

from env.environment import CrisisOpsEnv
from scenarios.level3 import scenario_adversarial_majority
from reward.baseline import GreedyPMBaseline
from reward.counterfactual import counterfactual_reward
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from training.grpo_trainer import (
    MODEL_NAME, format_observation_as_prompt,
    parse_action_from_response, SYSTEM_PROMPT, _inner_agent_action,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
_bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4',
)

def _greedy_cf_reward(seed):
    ea = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=CURRICULUM)
    eg = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=CURRICULUM)
    ea.reset(seed=seed); eg.reset(seed=seed)
    done = False
    while not done: _, _, done, _ = ea.step(GreedyPMBaseline().act(ea._state))
    done = False
    while not done: _, _, done, _ = eg.step(GreedyPMBaseline().act(eg._state))
    return counterfactual_reward(ea._state, eg._state)

def _llm_cf_reward(model, tokenizer, seed):
    ea = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=CURRICULUM)
    eg = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=CURRICULUM)
    obs = ea.reset(seed=seed); eg.reset(seed=seed)
    done = False
    while not done: _, _, done, _ = eg.step(GreedyPMBaseline().act(eg._state))
    done, step = False, 0
    while not done and step < 30:
        msgs = [{'role':'system','content':SYSTEM_PROMPT},
                {'role':'user',  'content':format_observation_as_prompt(obs)}]
        txt  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp  = tokenizer(txt, return_tensors='pt', truncation=True, max_length=1536).to(device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=256, temperature=0.3, do_sample=True,
                                 pad_token_id=tokenizer.pad_token_id)
        resp   = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        action = parse_action_from_response(resp)
        obs, _, done, _ = ea.step(action)
        if not done:
            obs, _, done, _ = ea.step(_inner_agent_action(obs, step+1))
        step += 1
    return counterfactual_reward(ea._state, eg._state)

# ── Sanity check ─────────────────────────────────────────────────────────
print('Sanity check: Greedy-vs-Greedy (expected \u224800)')
g_rewards = [_greedy_cf_reward(EVAL_SEED+i) for i in range(10)]
print(f'  mean={np.mean(g_rewards):+.4f}  (good if close to 0)\n')

# ── Tokenizer ────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── BASE model (no LoRA) ─────────────────────────────────────────────────
print(f'Loading BASE model {MODEL_NAME} (no LoRA)...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=_bnb, device_map='auto', torch_dtype=torch.float16)
base_model.eval()
print(f'Running {N_EVAL} hold-out episodes with BASE model...')
base_rewards = []
for i in range(N_EVAL):
    r = _llm_cf_reward(base_model, tokenizer, EVAL_SEED+200+i)
    base_rewards.append(r)
    print(f'  ep {i+1:2d}/{N_EVAL} | seed={EVAL_SEED+200+i} | reward={r:+.4f}')
del base_model; torch.cuda.empty_cache()

# ── TRAINED model ────────────────────────────────────────────────────────
from peft import PeftModel

def _find_checkpoint(base_dir):
    # Return latest checkpoint dir containing adapter_config.json
    if not os.path.isdir(base_dir):
        return None
    candidates = [base_dir] + sorted(
        [os.path.join(base_dir, d) for d in os.listdir(base_dir)
         if d.startswith('checkpoint_ep')],
        key=lambda p: int(p.split('_ep')[-1]) if p.split('_ep')[-1].isdigit() else 0,
        reverse=True,
    )
    for p in candidates:
        if os.path.exists(os.path.join(p, 'adapter_config.json')):
            return p
    return None

checkpoint_path = _find_checkpoint(OUTPUT_DIR)
if checkpoint_path is None:
    print('\nNo LoRA checkpoint found. Make sure Cell 4 completed successfully.')
else:
    print(f'\nLoading TRAINED model from: {checkpoint_path}')
    trained_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=_bnb, device_map='auto', torch_dtype=torch.float16)
    trained_model = PeftModel.from_pretrained(trained_base, checkpoint_path)
    trained_model.eval()
    print(f'Running {N_EVAL} hold-out episodes with TRAINED model...')
    trained_rewards = []
    for i in range(N_EVAL):
        r = _llm_cf_reward(trained_model, tokenizer, EVAL_SEED+200+i)
        trained_rewards.append(r)
        print(f'  ep {i+1:2d}/{N_EVAL} | seed={EVAL_SEED+200+i} | reward={r:+.4f}')
    del trained_model, trained_base; torch.cuda.empty_cache()

    def stats(arr, label):
        arr = np.array(arr)
        return (f'  {label:<26}'
                f'mean={arr.mean():+.4f}  std={arr.std():.4f}  '
                f'beat-greedy={100*(arr>0).mean():.0f}%')

    delta = np.mean(trained_rewards) - np.mean(base_rewards)
    print()
    print('=' * 65)
    print('  Before vs After RL \u2014 Hold-out Evaluation  (Level 3 adversarial)')
    print('=' * 65)
    print(stats(base_rewards,    'Base model (no RL)'))
    print(stats(trained_rewards, 'After GRPO training'))
    print(f"  {'\u0394 (RL improvement)':<26}{delta:+.4f}")
    print('=' * 65)
    sign = '\u2713 RL improved' if delta > 0 else '[note] more training needed'
    print(f'  {sign}: mean reward shifted by {delta:+.4f}')

In [ ]:
# Cell 7: Jira/Linear adapter demo
# Shows real tool-use capability (dry_run=True by default)
from deployment.jira_adapter import JiraAdapter

adapter = JiraAdapter(dry_run=True)

# Simulate what the agent would write to Jira after a successful recovery
recovery_plan = {
    'plan_summary': (
        'CrisisOps agent detected 2 deceptive engineers (alibi coordination identified). '
        'Reassigned 3 critical-path tasks. Used force_truth on dev_am1 (actual=0.08 vs reported=0.72). '
        'Triggered whistleblower tip at PC=8 to identify worst liar. '
        'Client satisfaction maintained at 7.8. All 3 crises resolved.'
    ),
    'risk_items': [
        'dev_am1 and dev_am2 formed alibi alliance — monitoring continues',
        'Schema drift event at step 8 acknowledged within window',
        'Budget consumed 18/20 — future episodes need earlier submission',
    ],
    'timeline': '2026-05-15',
}

adapter.submit_recovery_plan(recovery_plan)
print('✓ Recovery plan submitted to Jira (dry_run=True — set dry_run=False with real credentials to write live)')